**Imports and Setup**

In [72]:
import joblib
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import xgboost as xgb
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch will train on device: {device}")

PyTorch will train on device: cuda


**Load and Merge the Datasets**

In [73]:
print("Loading feature datasets...")

audio_df = pd.read_parquet("../../data/processed/wav2vec_features.parquet")
text_df = pd.read_parquet("../../data/processed/text_features.parquet")

merged_df = pd.merge(audio_df, text_df, on=['original_call_id', 'chunk_index', 'label'])
merged_df = merged_df.sort_values(by=['original_call_id', 'chunk_index']).reset_index(drop=True)

print(f"Audio shape: {audio_df.shape}")
print(f"Text shape: {text_df.shape}")
print(f"Final Merged Multimodal Dataset Shape: {merged_df.shape}")
merged_df.head(3)

Loading feature datasets...
Audio shape: (3958, 2307)
Text shape: (3958, 772)
Final Merged Multimodal Dataset Shape: (3958, 3076)


,original_call_id,chunk_index,label,w2v_dim_1,w2v_dim_2,w2v_dim_3,w2v_dim_4,w2v_dim_5,w2v_dim_6,w2v_dim_7,...,text_dim_760,text_dim_761,text_dim_762,text_dim_763,text_dim_764,text_dim_765,text_dim_766,text_dim_767,text_dim_768,chunk_text
0,sample_0,0,0,-0.006204,0.180686,0.281851,0.033207,0.301619,-0.197918,0.002960,...,-0.021365,0.022887,-0.050455,-0.003284,-0.041325,-0.011346,0.031532,0.040242,0.002005,Greetings. This is. Name. I finally got my han...
1,sample_0,1,0,0.011350,0.165507,0.276120,0.127721,0.298038,-0.156953,0.038472,...,-0.002989,0.031355,-0.045678,0.054876,-0.001477,0.016482,0.017949,0.014928,-0.044811,product. You mentioned last month. It is as go...
2,sample_0,2,0,0.044009,0.325490,0.056253,0.197733,0.502264,-0.088799,0.112194,...,-0.012205,0.006638,-0.068714,-0.032664,0.024762,0.017319,-0.028105,-0.004401,-0.045563,Let me know when you want to try it out.


**The 3D Grouping & Padding**

In [74]:
grouped = merged_df.groupby('original_call_id')

sequences = []
labels = []
seq_lengths = []

drop_cols = ['original_call_id', 'chunk_index', 'label', 'chunk_text']

for call_id, group in grouped:
    # Extract features for this specific call in chronological order
    features = group.drop(columns=drop_cols, errors='ignore').values
    label = group['label'].iloc[0] # The label is the same for the whole call
    
    sequences.append(torch.FloatTensor(features))
    labels.append(label)
    seq_lengths.append(len(features))

# Pad the sequences (adds zeros to short calls so everything is the same length)
X_padded = pad_sequence(sequences, batch_first=True, padding_value=0.0)
y = np.array(labels)
lengths = torch.LongTensor(seq_lengths)

print(f"Total Calls: {len(sequences)}")
print(f"3D Tensor Shape: {X_padded.shape} -> (Batch Size, Max Chunks, Features)")

Total Calls: 800
3D Tensor Shape: torch.Size([800, 21, 3072]) -> (Batch Size, Max Chunks, Features)


**Train/Test Split**

In [75]:
indices = np.arange(len(sequences))

train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=y)

X_train, y_train, len_train = X_padded[train_idx], y[train_idx], lengths[train_idx]
X_test, y_test, len_test = X_padded[test_idx], y[test_idx], lengths[test_idx]

print(f"Training Calls: {len(X_train)} | Testing Calls: {len(X_test)}")

Training Calls: 640 | Testing Calls: 160


**Model-1: Neural Network**

In [76]:
class SequenceFraudDataset(Dataset):
    def __init__(self, sequences, labels, lengths):
        self.sequences = sequences
        self.labels = torch.FloatTensor(labels).unsqueeze(1)
        self.lengths = lengths

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx], self.lengths[idx]

train_loader = DataLoader(SequenceFraudDataset(X_train, y_train, len_train), batch_size=32, shuffle=True)
test_loader = DataLoader(SequenceFraudDataset(X_test, y_test, len_test), batch_size=32, shuffle=False)

In [77]:
class LSTMStatefulClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, num_layers=1):
        super(LSTMStatefulClassifier, self).__init__()
        
        # Read the chunks sequentially
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )


    def forward(self, x, lengths):
        packed_input = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, (hn, cn) = self.lstm(packed_input)
        
        # hn[-1] grabs the hidden memory state from the VERY LAST valid chunk in the call
        last_hidden = hn[-1] 
        return self.classifier(last_hidden)


input_dim = X_padded.shape[2]
model_1 = LSTMStatefulClassifier(input_dim=input_dim).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_1.parameters(), lr=0.001, weight_decay=1e-4)

print("Initialized Stateful LSTM.")

Initialized Stateful LSTM.


In [ ]:
print("Training PyTorch Stateful LSTM Classifier...")
epochs = 50
for epoch in range(epochs):
    model_1.train()
    running_loss = 0.0
    
    for batch_seqs, batch_labels, batch_lens in train_loader:
        batch_seqs = batch_seqs.to(device)
        batch_labels = batch_labels.to(device)
        
        optimizer.zero_grad()
        # We pass both the data and the true lengths to the model
        outputs = model_1(batch_seqs, batch_lens) 
        loss = criterion(outputs, batch_labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    
    print(f"Epoch [{epoch+1:02d}/{epochs}] | Loss: {avg_loss:.4f}")

Training PyTorch Stateful LSTM Classifier...
Epoch [01/50] | Loss: 0.4200
Epoch [02/50] | Loss: 0.2307
Epoch [03/50] | Loss: 0.1611
Epoch [04/50] | Loss: 0.1658
Epoch [05/50] | Loss: 0.0949
Epoch [06/50] | Loss: 0.0759
Epoch [07/50] | Loss: 0.0579
Epoch [08/50] | Loss: 0.0533
Epoch [09/50] | Loss: 0.0411
Epoch [10/50] | Loss: 0.0381
Epoch [11/50] | Loss: 0.0590
Epoch [12/50] | Loss: 0.0281
Epoch [13/50] | Loss: 0.0124
Epoch [14/50] | Loss: 0.0078
Epoch [15/50] | Loss: 0.0177
Epoch [16/50] | Loss: 0.0122
Epoch [17/50] | Loss: 0.0113
Epoch [18/50] | Loss: 0.0225
Epoch [19/50] | Loss: 0.0891
Epoch [20/50] | Loss: 0.0165
Epoch [21/50] | Loss: 0.0288
Epoch [22/50] | Loss: 0.0432
Epoch [23/50] | Loss: 0.0140
Epoch [24/50] | Loss: 0.0370
Epoch [25/50] | Loss: 0.0372
Epoch [26/50] | Loss: 0.0382
Epoch [27/50] | Loss: 0.0365
Epoch [28/50] | Loss: 0.0487
Epoch [29/50] | Loss: 0.0392
Epoch [30/50] | Loss: 0.0339
Epoch [31/50] | Loss: 0.0104
Epoch [32/50] | Loss: 0.0042
Epoch [33/50] | Loss: 0.002

In [79]:
# Evaluation Phase
model_1.eval()
lstm_preds = []

print("\nEvaluating LSTM model...")
with torch.no_grad():
    for batch_seqs, _, batch_lens in test_loader:
        batch_seqs = batch_seqs.to(device)
        logits = model_1(batch_seqs, batch_lens)
        probs = torch.sigmoid(logits)
        preds = torch.round(probs).cpu().numpy()
        lstm_preds.extend(preds)

lstm_preds = np.array(lstm_preds).flatten()

print("\n" + "="*40)
print("MODEL: STATEFUL LSTM FUSION")
print("="*40)
print(f"Call-Level Accuracy: {accuracy_score(y_test, lstm_preds):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, lstm_preds, target_names=['Legitimate (0)', 'Scam (1)']))
print("Confusion Matrix:")
print(confusion_matrix(y_test, lstm_preds))


Evaluating LSTM model...

MODEL: STATEFUL LSTM FUSION
Call-Level Accuracy: 0.9625

Classification Report:
                precision    recall  f1-score   support

Legitimate (0)       0.97      0.95      0.96        80
      Scam (1)       0.95      0.97      0.96        80

      accuracy                           0.96       160
     macro avg       0.96      0.96      0.96       160
  weighted avg       0.96      0.96      0.96       160

Confusion Matrix:
[[76  4]
 [ 2 78]]


**Create the XGBoost Data**

In [80]:
print("Preparing 2D Mean-Pooled Data for XGBoost...")

xgb_features = []
xgb_labels = []

for call_id, group in grouped:
    features = group.drop(columns=drop_cols, errors='ignore').values
    label = group['label'].iloc[0]
    
    # Average all chunks together to create a single 3072-dimension vector for the whole call
    call_avg_features = np.mean(features, axis=0)
    
    xgb_features.append(call_avg_features)
    xgb_labels.append(label)

X_xgb = np.array(xgb_features)
y_xgb = np.array(xgb_labels)

X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(
    X_xgb, y_xgb, test_size=0.2, random_state=42, stratify=y_xgb
)

print(f"XGBoost Training Data Shape: {X_train_xgb.shape}")

Preparing 2D Mean-Pooled Data for XGBoost...
XGBoost Training Data Shape: (640, 3072)


**Model-2: XGBoost**

In [81]:
model_2 = xgb.XGBClassifier(
    n_estimators=100, 
    max_depth=4, 
    learning_rate=0.1, 
    random_state=42,
    eval_metric='logloss'
)

In [82]:
print("\nTraining XGBoost Classifier Baseline...")
model_2.fit(X_train_xgb, y_train_xgb)


Training XGBoost Classifier Baseline...


,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [84]:
print("Evaluating XGBoost model...")
xgb_preds = model_2.predict(X_test_xgb)

print("\n" + "="*40)
print("MODEL 2: XGBOOST ENSEMBLE")
print("="*40)
print(f"Accuracy: {accuracy_score(y_test_xgb, xgb_preds):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_xgb, xgb_preds, target_names=['Legitimate (0)', 'Scam (1)']))
print("Confusion Matrix:")
print(confusion_matrix(y_test_xgb, xgb_preds))

Evaluating XGBoost model...

MODEL 2: XGBOOST ENSEMBLE
Accuracy: 0.9688

Classification Report:
                precision    recall  f1-score   support

Legitimate (0)       0.97      0.96      0.97        80
      Scam (1)       0.96      0.97      0.97        80

      accuracy                           0.97       160
     macro avg       0.97      0.97      0.97       160
  weighted avg       0.97      0.97      0.97       160

Confusion Matrix:
[[77  3]
 [ 2 78]]


**Save Models**

In [86]:
models_dir = "../../models"
os.makedirs(models_dir, exist_ok=True)

pytorch_path = os.path.join(models_dir, "pytorch_fraud_model.pth")
torch.save(model_1.cpu().state_dict(), pytorch_path)

xgb_path = os.path.join(models_dir, "xgboost_fraud_model.joblib")
joblib.dump(model_2, xgb_path)

print(f"Serialization complete.")
print(f"XGBoost model saved to: {xgb_path}")
print(f"PyTorch weights saved to: {pytorch_path}")

Serialization complete.
XGBoost model saved to: ../../models\xgboost_fraud_model.joblib
PyTorch weights saved to: ../../models\pytorch_fraud_model.pth
